In [64]:
import re
from typing import Annotated, Literal

import pandas as pd
from pydantic import BaseModel, BeforeValidator, TypeAdapter

In [65]:
bracket_raw = pd.read_csv("00_bracket.csv")
lookup_raw = pd.read_csv("00_lookup.csv")
ratings_raw = pd.read_csv("01_ratings.csv")

In [66]:
ratings = (
    lookup_raw.merge(ratings_raw, how="left", left_on="eloratings_name", right_on="team")
    .set_index("fifa_code")["rating"]
    .to_dict()
)

In [67]:
def parse_team(s):
    if re.match(R"[WL]\d\d", s):
        return PrevMatch.from_string(s)
    return s


class PrevMatch(BaseModel):
    match_number: int
    team: Literal["W", "L"]

    @classmethod
    def from_string(cls, s: str):
        return PrevMatch(match_number=int(s[1:]), team=s[0])


class BracketMatch(BaseModel):
    match_number: int
    home: Annotated[str | PrevMatch, BeforeValidator(parse_team)]
    away: Annotated[str | PrevMatch, BeforeValidator(parse_team)]
    stage: str

In [68]:
matches = TypeAdapter(list[BracketMatch]).validate_python(bracket_raw.to_dict(orient="records"))

In [69]:
matches

[BracketMatch(match_number=73, home='RSA', away='CAN', stage='R32'),
 BracketMatch(match_number=74, home='GER', away='PAR', stage='R32'),
 BracketMatch(match_number=75, home='NED', away='MAR', stage='R32'),
 BracketMatch(match_number=76, home='BRA', away='JPN', stage='R32'),
 BracketMatch(match_number=77, home='FRA', away='SWE', stage='R32'),
 BracketMatch(match_number=78, home='CIV', away='NOR', stage='R32'),
 BracketMatch(match_number=79, home='MEX', away='ECU', stage='R32'),
 BracketMatch(match_number=80, home='ENG', away='COD', stage='R32'),
 BracketMatch(match_number=81, home='USA', away='BIH', stage='R32'),
 BracketMatch(match_number=82, home='BEL', away='SEN', stage='R32'),
 BracketMatch(match_number=83, home='POR', away='CRO', stage='R32'),
 BracketMatch(match_number=84, home='ESP', away='AUT', stage='R32'),
 BracketMatch(match_number=85, home='SUI', away='ALG', stage='R32'),
 BracketMatch(match_number=86, home='ARG', away='CPV', stage='R32'),
 BracketMatch(match_number=87, hom